# Reconnaissance d'expressions faciales — FER2013

**Projet Deep Learning** — Étudiant A / Étudiant B

**Objectif :** entraîner et comprendre un modèle qui reconnaît 7 expressions faciales
(angry, disgust, fear, happy, neutral, sad, surprise) à partir d'images de visages 48×48 en niveaux de gris.

**Dataset :** FER2013 — Kaggle `msambare/fer2013`

**Plan du notebook** (construit progressivement, une cellule par micro-étape) :
1. Exploration et préparation du dataset
2. Baseline dense
3. CNN
4. Entraînement
5. Évaluation et analyse des erreurs
6. Expériences (≥ 3)


### Étape 0 — Téléchargement des données (si nécessaire)

**Pourquoi ?** `data/` n'est pas commité dans Git (201 Mo, ~35 000 fichiers). Sans cette cellule, un `git clone` ne contient aucun image et il faudrait télécharger FER2013 à la main sur chaque PC ou sur Colab.

**Concept clé : la cellule est *idempotente*.** La relancer 10 fois donne exactement le même résultat qu'une seule fois : si `data/fer2013/train` et `data/fer2013/test` existent déjà, elle n'imprime qu'un message et ne télécharge rien ; sinon elle télécharge le dataset via `kagglehub` (bibliothèque officielle de Kaggle) et copie les splits au bon endroit.

Le script commence par trouver la **racine du projet** (le dossier qui contient `.gitignore`) : les chemins `data/fer2013` fonctionnent donc que le notebook soit lancé depuis la racine ou depuis `notebooks/`.


In [ ]:
# Étape 0 : télécharger FER2013 uniquement s'il manque (cellule idempotente)
import os
import shutil
from pathlib import Path

# 1) Racine du projet = dossier qui contient .gitignore.
#    Permet de lancer le notebook depuis la racine OU depuis notebooks/.
racine = Path.cwd()
while racine != racine.parent and not (racine / ".gitignore").exists():
    racine = racine.parent
if not (racine / ".gitignore").exists():
    raise FileNotFoundError("Racine du projet (.gitignore) introuvable.")

# On se place à la racine : les étapes 1a à 1c utilisent des chemins
# relatifs "data/fer2013" sans changer de dossier.
os.chdir(racine)

DATA_DIR = Path("data/fer2013")

# 2) Les deux splits sont déjà là ? -> rien à télécharger.
if (DATA_DIR / "train").is_dir() and (DATA_DIR / "test").is_dir():
    print(f"Données déjà présentes dans {DATA_DIR} — aucun téléchargement.")
else:
    import kagglehub  # seule dépendance de cette cellule (sur Colab: déjà installée)

    print("Téléchargement du dataset msambare/fer2013 via kagglehub…")
    source = Path(kagglehub.dataset_download("msambare/fer2013"))

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for split in ("train", "test"):
        destination = DATA_DIR / split
        if not destination.exists():  # jamais réécrit si déjà copié
            shutil.copytree(source / split, destination)
    print(f"Dataset copié dans {DATA_DIR.resolve()}")

# 3) Vérification : nombre d'images par split (attendu : 28 709 / 7 178)
for split in ("train", "test"):
    n_images = sum(1 for f in (DATA_DIR / split).rglob("*") if f.is_file())
    print(f"{split:5s} : {n_images} images")


Avant de charger les données, on inspecte l'arborescence de `data/fer2013` : chaque dossier de classe sert d'étiquette (label), c'est cette structure que Keras lira.

In [ ]:
# Micro-étape 1a : explorer la structure du dataset FER2013
from pathlib import Path

DATA_DIR = Path("data/fer2013")  # chemin relatif à la racine du projet

for split in ["train", "test"]:  # les deux splits du dataset
    total = 0
    # on trie les classes par nom pour un affichage stable et lisible
    for classe in sorted(p for p in (DATA_DIR / split).iterdir() if p.is_dir()):
        n = len(list(classe.glob("*.jpg")))  # nombre d'images .jpg dans ce dossier de classe
        total += n
        print(f"{split}/{classe.name} : {n}")
    print(f"{split} total : {total}\n")  # total du split

### Étape 1b — Regarder les images et vérifier le format

**Pourquoi ?** Un modèle apprend à partir de pixels : s'il reçoit des images de tailles ou de modes différents, la couche `Flatten` produira des vecteurs de longueurs différentes et l'entraînement échouera (ou buggera silencieusement).

On fait donc deux choses :
1. **On affiche une grille de 7 classes × 5 images** tirées de `data/fer2013/train/<classe>/` : on vérifie *à l'œil* que les dossiers contiennent bien des visages et que les classes sont cohérentes (utile aussi pour l'oral : montrer des exemples réels).
2. **On vérifie le format sur *toutes* les images** (train + test) avec PIL : taille `(48, 48)`, mode `L` (niveaux de gris) et extension `.jpg`. On liste les exceptions plutôt que de les faire échouer en silence.

> Librairies utilisées uniquement : `PIL`, `matplotlib` et `os` — aucune dépendance supplémentaire.


In [ ]:
# Micro-étape 1b : exemples d'images + vérification du format (taille, mode, extension)
import os
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = os.path.join("data", "fer2013")  # racine du dataset (chemin relatif à la racine du projet)
N_EXEMPLES = 5  # nombre d'images affichées par classe
# les 7 classes dans l'ordre canonique : cet ordre sera repris pour l'encodage one-hot (étape 1c)
CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

# --- 1) grille 7 lignes (classes) x 5 colonnes (exemples) -------------------
fig, axes = plt.subplots(len(CLASSES), N_EXEMPLES, figsize=(10, 14))
for ligne, classe in enumerate(CLASSES):
    dossier = os.path.join(DATA_DIR, "train", classe)
    # on trie les noms de fichiers pour que la grille soit reproductible à chaque exécution
    fichiers = sorted(f for f in os.listdir(dossier) if f.endswith(".jpg"))[:N_EXEMPLES]
    for col, nom in enumerate(fichiers):
        with Image.open(os.path.join(dossier, nom)) as img:  # le `with` ferme l'image automatiquement
            axes[ligne, col].imshow(img, cmap="gray")  # cmap="gray" car les images sont en niveaux de gris
        axes[ligne, col].set_xticks([])  # on masque les graduations : on veut voir l'image, pas les axes
        axes[ligne, col].set_yticks([])
    # nom de la classe écrit à gauche de la ligne (une seule fois)
    axes[ligne, 0].set_ylabel(classe, rotation=0, ha="right", va="center")

fig.suptitle("FER2013 — 5 exemples par classe (split train)", fontsize=14)
plt.tight_layout()
plt.show()

# --- 2) vérification du format sur TOUTES les images (train + test) --------
attendu = ((48, 48), "L", ".jpg")  # taille, mode PIL (L = niveaux de gris), extension
stats_taille, stats_mode, stats_ext = {}, {}, {}
anomalies = []
n_images = 0

for split in ["train", "test"]:
    racine_split = os.path.join(DATA_DIR, split)
    for classe in sorted(os.listdir(racine_split)):
        dossier = os.path.join(racine_split, classe)
        if not os.path.isdir(dossier):  # on ignore les fichiers éventuels à la racine du split
            continue
        for nom in sorted(os.listdir(dossier)):
            chemin = os.path.join(dossier, nom)
            n_images += 1
            ext = os.path.splitext(nom)[1].lower()
            try:
                with Image.open(chemin) as img:
                    taille, mode = img.size, img.mode
            except OSError:  # fichier corrompu ou illisible : on le note et on passe au suivant
                anomalies.append((chemin, "illisible", "illisible", ext))
                continue
            # on compte les valeurs observées pour pouvoir afficher un bilan honnête
            stats_taille[taille] = stats_taille.get(taille, 0) + 1
            stats_mode[mode] = stats_mode.get(mode, 0) + 1
            stats_ext[ext] = stats_ext.get(ext, 0) + 1
            if (taille, mode, ext) != attendu:
                anomalies.append((chemin, taille, mode, ext))

print(f"Images vérifiées : {n_images} (train + test)")
print(f"Tailles observées    : {stats_taille}")
print(f"Modes PIL observés   : {stats_mode}")
print(f"Extensions observées : {stats_ext}")
if anomalies:
    # critère de validation non rempli : on liste les exceptions au lieu de les cacher
    print(f"\n{len(anomalies)} exception(s) trouvée(s) sur {n_images} images :")
    for chemin, taille, mode, ext in anomalies[:20]:  # on n'affiche que les 20 premières
        print(f"  - {chemin} -> taille {taille}, mode {mode}, extension {ext}")
    if len(anomalies) > 20:
        print(f"  ... et {len(anomalies) - 20} autre(s)")
else:
    print(f"\nFormat : 48×48, mode L, .jpg — vérifié sur {n_images} images, aucune exception.")


### Étape 1c — Construire les datasets d'entraînement, de validation et de test

**Pourquoi ?** Jusqu'ici on a seulement *regardé* les fichiers. Le modèle a besoin d'un `tf.data.Dataset` : un flux de lots d'images + d'étiquettes, prêt pour `model.fit()`.

Quatre points à comprendre :

1. **Normalisation** — les pixels sont des entiers entre 0 et 255. On les divise par 255 pour les ramener dans `[0, 1]` : des entrées petites et à la même échelle rendent l'entraînement plus stable et, en général, plus rapide.
2. **Encodage one-hot** — `label_mode="categorical"` transforme l'étiquette `"happy"` en vecteur `[0, 0, 0, 1, 0, 0, 0]` de 7 valeurs (la position du 1 suit l'ordre de `CLASS_NAMES`). C'est la forme attendue par une sortie `softmax` avec la perte `categorical_crossentropy`.
3. **Validation** — 20 % de `data/fer2013/train` est mis de côté (5 741 images) pour mesurer la généralisation pendant l'entraînement et repérer le surapprentissage. Ces images n'entraînent jamais les poids : elles servent seulement à décider quand s'arrêter (`early stopping`, choix du meilleur modèle).
4. **Test** — `data/fer2013/test` (7 178 images) reste intact, sans mélange ni découpe : il est réservé à l'évaluation finale, une seule fois, à la toute fin du projet. Si on l'utilisait pendant le développement, on « tricherait » sans le savoir.

Rappel de l'étape 1b : toutes les images font déjà 48×48 en niveaux de gris, donc la forme d'une image est `(48, 48, 1)` et aucune redimensionnement n'est nécessaire.


In [ ]:
# Micro-étape 1c : construire les datasets TensorFlow (train / validation / test)
import os
from tensorflow import keras

# Les 7 classes, dans le même ordre que l'étape 1b.
# Cet ordre fixe la position du 1 dans le vecteur one-hot :
# 0 = angry, 1 = disgust, 2 = fear, 3 = happy, 4 = neutral, 5 = sad, 6 = surprise.
CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

BATCH_SIZE = 64            # nombre d'images par lot
IMAGE_SIZE = (48, 48)      # taille déjà vérifiée à l'étape 1b
VALIDATION_SPLIT = 0.2     # 20 % du dossier train est réservé à la validation

# --- 1) train + validation : une seule passe sur data/fer2013/train ---
# subset="both" renvoie directement un tuple (train_ds, val_ds).
# seed=42 rend la découpe reproductible : la même image tombe toujours du même côté.
# color_mode="grayscale"  -> 1 seul canal, donc une forme (48, 48, 1).
# label_mode="categorical" -> étiquette encodée en one-hot (vecteur de 7 valeurs).
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    os.path.join("data", "fer2013", "train"),
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    seed=42,
)

# --- 2) test : data/fer2013/test, utilisé tel quel, sans découpe ---
# shuffle=False garde l'ordre des fichiers, pour une évaluation stable et reproductible.
# Pas de validation_split : ce dossier sert uniquement à l'évaluation finale.
test_ds = keras.utils.image_dataset_from_directory(
    os.path.join("data", "fer2013", "test"),
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# --- 3) nombre d'images réellement présent dans chaque ensemble ---
# Chaque dataset conserve la liste de ses fichiers dans l'attribut .file_paths.
# On lit ces listes maintenant : le .map de l'étape suivante crée un nouveau dataset,
# qui ne possède plus cet attribut.
print("Images en train      :", len(train_ds.file_paths))      # attendu : 22968
print("Images en validation :", len(val_ds.file_paths))        # attendu : 5741
print("Images en test       :", len(test_ds.file_paths))       # attendu : 7178

# --- 4) normalisation : les pixels passent de l'échelle [0, 255] à [0, 1] ---
# Fonction nommée plutôt qu'un lambda : plus facile à lire et à expliquer ligne par ligne.
def normaliser(images, labels):
    return images / 255.0, labels

# .map applique la fonction à chaque lot du dataset, au fur et à mesure de la lecture.
train_ds = train_ds.map(normaliser)
val_ds = val_ds.map(normaliser)
test_ds = test_ds.map(normaliser)

# --- 5) contrôle : forme d'un lot et valeurs des pixels ---
# next(iter(train_ds)) lit le premier lot : 64 images et leurs étiquettes.
images_batch, labels_batch = next(iter(train_ds))
print("Forme du lot d'images  :", images_batch.shape)   # attendu : (64, 48, 48, 1)
print("Forme du lot de labels :", labels_batch.shape)   # attendu : (64, 7)
print("Valeur min des pixels  :", float(images_batch.numpy().min()))  # entre 0 et 1
print("Valeur max des pixels  :", float(images_batch.numpy().max()))  # entre 0 et 1
